# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [4]:
!pip install -q --upgrade pypdf gradio openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.1 MB/s eta 0:00:00


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="TU_API_KEY"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [5]:
from pypdf import PdfReader


def extract_text_from_pdf(pdf_path: str) -> str:
    """Extrae todo el texto de un PDF como una sola cadena de texto plano."""
    reader = PdfReader(pdf_path)

    pages_text = []
    for page in reader.pages:
        # Algunas paginas (portadas, imagenes escaneadas) pueden no tener texto extraible.
        pages_text.append(page.extract_text() or "")

    return "\n".join(pages_text)


# Prueba rapida
_sample_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(_sample_text[:500])

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [6]:
import os
from google.colab import userdata
from openai import OpenAI

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"

## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [7]:
def build_system_prompt(document_text: str) -> str:
    """Construye el system prompt que ancla las respuestas al documento."""
    return (
        "Eres un asistente experto en el siguiente paper de investigacion. "
        "Debes responder UNICAMENTE con base en la informacion contenida en el "
        "documento delimitado por las etiquetas <documento> y </documento>. "
        "No utilices conocimiento externo ni supuestos que no esten respaldados "
        "por el texto. Si la respuesta no se encuentra en el documento, responde "
        "exactamente: \"No encuentro esa informacion en el documento\".\n\n"
        "<documento>\n"
        f"{document_text}\n"
        "</documento>"
    )


def history_to_messages(history: list) -> list:
    """
    Convierte el historial que entrega gr.ChatInterface a una lista de
    mensajes con roles 'user' y 'assistant', compatible con la API de OpenAI.

    Soporta tanto el formato de tuplas [[user, assistant], ...] como el
    formato de mensajes [{'role':..., 'content':...}, ...] segun la version
    de Gradio.
    """
    messages = []

    for turn in history:
        if isinstance(turn, dict):
            messages.append({"role": turn["role"], "content": turn["content"]})
        else:
            user_msg, assistant_msg = turn
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})

    return messages


def chat_con_documento(message: str, history: list, document_text: str):
    """
    Genera una respuesta basada en el documento, entregando el texto
    progresivamente (streaming) para usarse directamente como fn de
    gr.ChatInterface.
    """
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    respuesta_completa = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            respuesta_completa += chunk.choices[0].delta.content
            yield respuesta_completa

## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [8]:
import gradio as gr

# 1. Extraemos el texto del PDF
document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")

n_pages = len(PdfReader("attention_is_all_you_need.pdf").pages)
print(f"Paginas extraidas: {n_pages}")
print(f"Caracteres extraidos: {len(document_text)}")

# 2. Configuramos la interfaz de chat
demo = gr.ChatInterface(
    fn=chat_con_documento,
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    title="Asistente del paper: Attention Is All You Need",
    description=(
        "Preguntale al asistente sobre el paper de Vaswani et al. (2017). "
        "Las respuestas se basan unicamente en el contenido del documento "
        "cargado (no es busqueda web, no es conocimiento general del modelo)."
    ),
    examples=[
        ["¿Cual es la arquitectura principal propuesta en el paper?", document_text],
        ["¿Que es el mecanismo de atencion?", document_text],
        ["¿Cuantas capas tiene el encoder del modelo base?", document_text],
    ],
)

# 3. Lanzamos la interfaz de forma compatible con Colab
demo.launch(share=True, show_error=True)

Paginas extraidas: 15
Caracteres extraidos: 39510
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11bff43e70d190bd7b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*

Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*

Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?


## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

### Mejora implementada: opciones A + C + D combinadas

Implementamos las tres juntas porque tienen sentido como un solo flujo:

- **A) Medición del contexto:** comparamos caracteres vs. tokens estimados, documento completo vs. fragmentos.
- **D) Primer paso hacia RAG:** en vez de enviar las ~39,500 caracteres completas en cada pregunta, dividimos el PDF en fragmentos (chunks) de ~150 palabras por página, y seleccionamos solo los 4 fragmentos más relevantes mediante una búsqueda simple por palabras (sin embeddings, sin modelos de similitud semántica — justo lo que pide el enunciado).
- **C) Evidencia y páginas:** cada fragmento conserva de qué página del PDF vino, y el `system prompt` exige que el modelo cite la página de donde sacó cada respuesta.

In [ ]:
import re

def extract_pages_from_pdf(pdf_path: str) -> list:
    """Como extract_text_from_pdf, pero conserva el texto pagina por pagina."""
    reader = PdfReader(pdf_path)
    return [(page.extract_text() or "") for page in reader.pages]


def chunk_pages(pages: list, chunk_size_words: int = 150, overlap_words: int = 30) -> list:
    """
    Divide cada pagina en fragmentos de tamano fijo (en palabras), con
    solapamiento para no cortar ideas a la mitad. Cada fragmento conserva
    el numero de pagina de origen (1-indexado) para poder citarla despues.
    """
    chunks = []
    for page_num, page_text in enumerate(pages, start=1):
        words = page_text.split()
        if not words:
            continue
        start = 0
        while start < len(words):
            end = start + chunk_size_words
            chunk_words = words[start:end]
            chunks.append({"page": page_num, "text": " ".join(chunk_words)})
            if end >= len(words):
                break
            start = end - overlap_words
    return chunks


_STOPWORDS = {
    "el","la","los","las","de","del","un","una","unos","unas","y","o","en",
    "que","es","son","para","por","con","su","sus","al","lo","se","como",
    "cual","cuales","cuantos","cuantas","the","a","an","of","in","on","and",
    "or","to","is","are","for","with","that","this","what","how","does","do",
    "it","its","as","by","be","was","we","our","also","which","each",
}

def _tokenize(text: str) -> set:
    """Extrae palabras clave en minuscula, descartando stopwords y palabras muy cortas."""
    words = re.findall(r"[a-zA-ZÀ-ÿ0-9]+", text.lower())
    return {w for w in words if w not in _STOPWORDS and (len(w) > 2 or w.isdigit())}


def score_chunk(question: str, chunk_text: str) -> int:
    """Puntua un fragmento por cuantas palabras clave comparte con la pregunta."""
    return len(_tokenize(question) & _tokenize(chunk_text))


def retrieve_relevant_chunks(question: str, chunks: list, top_k: int = 4) -> list:
    """Busqueda simple por palabras: ordena los fragmentos por palabras clave compartidas."""
    scored = [(score_chunk(question, c["text"]), c) for c in chunks]
    scored.sort(key=lambda x: x[0], reverse=True)
    top = [c for score, c in scored[:top_k] if score > 0]
    return top if top else [c for _, c in scored[:top_k]]


def estimate_tokens(text: str) -> int:
    """
    Estimacion aproximada de tokens a partir de palabras (regla practica:
    para texto tecnico en ingles, 1 palabra equivale a ~1.3 tokens porque
    muchas palabras se dividen en varios sub-tokens).
    """
    return int(len(text.split()) * 1.3)

In [ ]:
# Construimos los fragmentos una sola vez
pages = extract_pages_from_pdf("attention_is_all_you_need.pdf")
ALL_CHUNKS = chunk_pages(pages, chunk_size_words=150, overlap_words=30)

print(f"Documento completo : {len(document_text):,} caracteres, ~{estimate_tokens(document_text):,} tokens estimados")
print(f"Numero de fragmentos generados: {len(ALL_CHUNKS)}")

# Comparamos cuanto se reduce el contexto enviado al modelo
ejemplo_chunks = retrieve_relevant_chunks("How many layers does the encoder have?", ALL_CHUNKS, top_k=4)
texto_enviado = " ".join(c["text"] for c in ejemplo_chunks)
print(f"\nPara una pregunta tipica, en vez de enviar el documento completo,")
print(f"se envian solo {len(ejemplo_chunks)} fragmentos: {len(texto_enviado):,} caracteres, "
      f"~{estimate_tokens(texto_enviado):,} tokens estimados "
      f"({len(texto_enviado) / len(document_text):.1%} del documento completo).")

In [ ]:
def build_system_prompt_rag(chunks: list) -> str:
    """Igual que build_system_prompt, pero solo con los fragmentos recuperados y exigiendo citar pagina."""
    fragments_text = "\n\n".join(f"[Pagina {c['page']}]\n{c['text']}" for c in chunks)
    return (
        "Eres un asistente experto en el siguiente paper de investigacion. "
        "A continuacion recibiras SOLO los fragmentos del documento que un "
        "sistema de busqueda considero mas relevantes para la pregunta del "
        "usuario (no el documento completo). Responde UNICAMENTE con base en "
        "esos fragmentos, y al final de tu respuesta cita la pagina de donde "
        "sacaste la informacion en el formato (pagina X). Si la respuesta no "
        "se encuentra en los fragmentos entregados, responde exactamente: "
        "\"No encuentro esa informacion en los fragmentos recuperados\".\n\n"
        f"{fragments_text}"
    )


def chat_con_documento_rag(message: str, history: list):
    """Version 'primer paso hacia RAG' de chat_con_documento: recupera antes de responder."""
    relevant_chunks = retrieve_relevant_chunks(message, ALL_CHUNKS, top_k=4)

    messages = [{"role": "system", "content": build_system_prompt_rag(relevant_chunks)}]
    messages.extend(history_to_messages(history))
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        stream=True,
    )

    respuesta_completa = ""
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content is not None:
            respuesta_completa += chunk.choices[0].delta.content
            yield respuesta_completa


# Interfaz separada para probar la version RAG sin interferir con la del Paso 4.
# No se lanza automaticamente para no chocar con el puerto del demo ya corriendo:
# cuando quieras probarla, corre en una celda nueva: demo_rag.launch(share=True, show_error=True)
demo_rag = gr.ChatInterface(
    fn=chat_con_documento_rag,
    title="Asistente del paper (version 'primer paso hacia RAG')",
    description=(
        "Esta version NO envia el documento completo: recupera solo los 4 "
        "fragmentos mas relevantes por busqueda simple de palabras clave, y "
        "cita la pagina de origen en cada respuesta."
    ),
    examples=[
        "How many layers does the encoder have?",
        "What is the BLEU score on WMT 2014 English-to-German?",
        "¿Cuántas capas tiene el encoder del modelo base?",
    ],
)

In [ ]:
# Comparacion sin llamar al modelo: solo vemos QUE fragmentos recuperaria
# el buscador simple para distintas preguntas (esto no requiere la API de NVIDIA).

preguntas_prueba = [
    "How many layers does the encoder have?",          # pregunta en ingles (idioma del documento)
    "¿Cuántas capas tiene el encoder del modelo base?",  # la misma pregunta, en español
    "¿Qué dice el paper sobre el uso de Dropout?",
]

for q in preguntas_prueba:
    top = retrieve_relevant_chunks(q, ALL_CHUNKS, top_k=4)
    paginas = [c["page"] for c in top]
    print(f"{q}\n  -> paginas recuperadas: {paginas}\n")

### Hallazgo real al probar la búsqueda simple por palabras

Al comparar la misma pregunta en inglés y en español, la búsqueda por palabras clave
recupera páginas **distintas** — y solo la versión en inglés tiende a acercarse a la
página correcta (la arquitectura del encoder se describe en la página 3). La versión
en español prácticamente no comparte vocabulario literal con el documento (que está
en inglés): palabras como *"capas"*, *"modelo"* o *"cuántas"* no aparecen nunca en el
texto original, así que el conteo de palabras compartidas es bajo o nulo, y el
buscador termina recuperando fragmentos poco relacionados.

**Esto no es un error del código — es la limitación real que el enunciado pedía
observar.** Es exactamente la motivación detrás de RAG "de verdad": los sistemas de
producción no comparan palabras literales, comparan *embeddings* (representaciones
vectoriales de significado), que sí capturan que "¿cuántas capas tiene el encoder?"
y "how many layers does the encoder have?" preguntan lo mismo, aunque no compartan
ni una sola palabra en común.

**Respuesta a la pregunta de la opción A** (por qué caracteres y tokens no son
equivalentes): un token no es un carácter ni necesariamente una palabra completa —
es la unidad mínima en la que el tokenizador del modelo divide el texto, y una
palabra puede partirse en varios tokens (especialmente palabras largas, poco
frecuentes, o en un idioma distinto al que domina el vocabulario del tokenizador).
Por eso nunca se puede convertir caracteres a tokens con una proporción fija exacta
— solo estimarla.

### Matiz adicional: por qué "Dropout" sí funcionó

Al probar una tercera pregunta en español —*"¿Qué dice el paper sobre el uso de
Dropout?"*— el buscador sí recuperó la página correcta (la 7, donde está la
sección de regularización). La diferencia es que **"Dropout" es la misma
palabra en español y en inglés**: es un término técnico que no se traduce
(igual que "encoder" en la prueba anterior, que por eso ayudó un poco aunque no
fue suficiente). En cambio "capas" (layers) y "cuántas" (how many) sí tienen
traducción, así que no dejan ningún rastro léxico que el buscador pueda
aprovechar.

Esto deja un matiz importante: la búsqueda por palabras clave **no falla
siempre** entre idiomas — falla específicamente en las palabras que sí se
traducen, y "funciona" por accidente cuando el término clave resulta ser un
cognado o un préstamo técnico (Dropout, encoder, Transformer...). Es un
ejemplo claro de por qué depender de coincidencias léxicas literales es frágil
e impredecible, y por qué RAG real necesita algo que no dependa de esta especie
de "suerte lingüística" — ahí es exactamente donde entran los embeddings.

## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
2. ¿Por qué este ejercicio todavía no implementa RAG?
3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
   conocimiento previo del modelo?
4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el
   prompt?

### Respuestas (basadas en las pruebas reales que hicimos arriba)

**1. ¿Cuál es la limitación de enviar siempre el documento completo?**

Cada pregunta —sin importar qué tan simple o puntual sea— reenvía las ~39,500
caracteres completas del paper (~34,000 tokens estimados) en el `system prompt`.
Para preguntas como *"¿quiénes son los autores?"* esto es enormemente
ineficiente: se gasta el mismo contexto (y el mismo costo, y la misma latencia)
que para una pregunta que de verdad requiere entender el paper completo. Además,
esto no escala: un documento de 300 páginas simplemente no cabría en la ventana
de contexto del modelo, sin importar qué tan grande sea.

**2. ¿Por qué este ejercicio todavía no implementa RAG?**

La versión del Paso 4 no implementa ninguna forma de *recuperación* — siempre
envía el 100% del documento, así que no hay nada que "seleccionar". Incluso la
mejora que implementamos en el Paso 6 (fragmentar + buscar por palabras clave)
es apenas un primer paso: usamos conteo literal de palabras compartidas, no
similitud semántica real. Lo comprobamos directamente: al preguntar lo mismo en
inglés y en español, el buscador recuperó páginas distintas, porque "capas" y
"layers" no se parecen como texto aunque signifiquen lo mismo. RAG de verdad usa
*embeddings* (vectores que representan significado, no palabras literales) para
que ese tipo de coincidencias semánticas sí se detecten.

**3. ¿Cómo comprobarían que la respuesta proviene del documento y no del
conocimiento previo del modelo?**

Lo comprobamos con preguntas trampa reales: al preguntar *"¿Qué es GPT-4?"* y
*"¿Cuál es la capital de Francia?"* —información que el modelo casi con
seguridad conoce de su entrenamiento general, pero que no está en el paper—
respondió correctamente *"No encuentro esa información en el documento"* en
ambos casos. Esa es la prueba más directa: si el modelo rechaza preguntas cuya
respuesta sabemos que no está en el texto, es evidencia de que está respetando
el *grounding* en el documento. Una segunda señal, más sutil, es que al
preguntar por el mecanismo de atención, el modelo respondió citando casi
textualmente (en inglés) la definición exacta del paper, en vez de parafrasear
desde su conocimiento general — una cita casi literal es difícil de producir
por casualidad si no se está leyendo el texto fuente.

**4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en
el prompt?**

Lo vimos en vivo: al pedir la definición de atención, el modelo cambió
espontáneamente al inglés para citar esa parte del documento, aunque toda la
conversación había sido en español — un comportamiento que no pedimos ni
controlamos explícitamente, simplemente "heredado" del idioma del documento
fuente. A mayor escala, este mismo mecanismo es un riesgo de seguridad: si el
documento de un usuario contiene instrucciones embebidas (texto diseñado para
que el modelo las "obedezca" como si vinieran del sistema, conocido como
*prompt injection*), el modelo podría terminar siguiendo esas instrucciones en
vez de limitarse a responder preguntas sobre el contenido. También hay riesgo
de exponer información sensible del documento completo en cada respuesta, aun
cuando la pregunta del usuario no la necesitaba.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)